# 04 — Cruzamentos: chamados, chuva, logradouros e bairros
**Projeto de extensão: Ciência de Dados para Cidades Inteligentes**

Último notebook da análise exploratória. Ele **não consulta de novo** as tabelas grandes: lê os arquivos que os notebooks 01, 02 e 03 salvaram em `data/processed/`.

**Antes de rodar este notebook, rode os três anteriores.**

## 0. Preparando o ambiente
Rodo o notebook de autenticação, que instala as bibliotecas, faz os imports, define o `BILLING_PROJECT_ID` e testa a conexão com o BigQuery. Assim o ID do projeto fica escrito num lugar só.

In [ ]:
%run ./00_autenticacao_bigquery.ipynb

## Carregando os dados salvos pelos notebooks anteriores
Se algum arquivo estiver faltando, a célula para com uma mensagem dizendo qual notebook rodar.

As séries mensais foram salvas com uma data no lugar do mês, então transformo de volta em mês (`to_period`) para que as duas séries se alinhem na seção 5.4.

In [ ]:
from pathlib import Path

PASTA_PROCESSED = Path("../data/processed")

arquivos_necessarios = {
    "chamados_manejo_2021.parquet": "01_eda_chamados_1746",
    "chamados_por_mes.parquet": "01_eda_chamados_1746",
    "chuva_por_mes.parquet": "02_eda_clima",
    "logradouros_trechos.parquet": "03_eda_logradouros",
    "bairros.parquet": "03_eda_logradouros",
}
faltando = {arquivo: notebook for arquivo, notebook in arquivos_necessarios.items()
            if not (PASTA_PROCESSED / arquivo).exists()}
if faltando:
    raise FileNotFoundError(f"Arquivos faltando (rode o notebook indicado): {faltando}")

chamados = pd.read_parquet(PASTA_PROCESSED / "chamados_manejo_2021.parquet")
logradouros = pd.read_parquet(PASTA_PROCESSED / "logradouros_trechos.parquet")
bairros = pd.read_parquet(PASTA_PROCESSED / "bairros.parquet")

por_mes = pd.read_parquet(PASTA_PROCESSED / "chamados_por_mes.parquet")["chamados"]
por_mes.index = por_mes.index.to_period("M")

chuva_mes = pd.read_parquet(PASTA_PROCESSED / "chuva_por_mes.parquet")["chuva_mm"]
chuva_mes.index = chuva_mes.index.to_period("M")

print("Chamados:", len(chamados), "| Trechos de rua:", len(logradouros), "| Bairros:", len(bairros))
print("Meses com chamados:", len(por_mes), "| Meses com chuva:", len(chuva_mes))

# 5. Juntando as tabelas (primeiros testes)
Agora que conheço cada tabela, faço os primeiros cruzamentos. Ainda não é a análise final: a ideia é ver se as tabelas "conversam" entre si.

## 5.1 Colocando o nome do bairro nos chamados
Uso o `merge` do pandas (parecido com o `LEFT JOIN` do SQL). Antes, transformo os ids em texto nas duas tabelas, para garantir que estão no mesmo formato.

In [ ]:
chamados["id_bairro_texto"] = chamados["id_bairro"].astype("string")
bairros["id_bairro_texto"] = bairros["id_bairro"].astype("string")

chamados_com_bairro = chamados.merge(
    bairros[["id_bairro_texto", "nome"]].rename(columns={"nome": "nome_bairro"}),
    on="id_bairro_texto",
    how="left",
)

sem_id = chamados_com_bairro["id_bairro"].isnull().sum()
id_sem_nome = (chamados_com_bairro["id_bairro"].notnull() & chamados_com_bairro["nome_bairro"].isnull()).sum()
print("Chamados sem id de bairro:", sem_id)
print("Chamados com id de bairro que não existe na tabela de bairros:", id_sem_nome)

chamados_com_bairro["nome_bairro"].value_counts().head(15).sort_values().plot(kind="barh", figsize=(9, 6), color="forestgreen")
plt.title("Top 15 bairros com mais chamados de Manejo Arbóreo (2021 em diante)")
plt.show()

## 5.2 O ranking muda se eu levar em conta o tamanho do bairro?
Bairros grandes, como Campo Grande, têm mais ruas e por isso mais chances de ter chamados. Divido os chamados pelo número de trechos de rua do bairro (seção 3.6) e calculo **chamados a cada 100 trechos**. Deixo de fora bairros com menos de 100 trechos, porque com poucos trechos qualquer chamado vira um número enorme.

In [ ]:
chamados_por_bairro = chamados_com_bairro.groupby("id_bairro_texto").size()
trechos_bairro = logradouros.groupby("id_bairro").size()
trechos_bairro.index = trechos_bairro.index.astype("string")

comparacao = pd.DataFrame({"chamados": chamados_por_bairro, "trechos": trechos_bairro}).dropna()
comparacao = comparacao[comparacao["trechos"] >= 100]
comparacao["chamados_por_100_trechos"] = 100 * comparacao["chamados"] / comparacao["trechos"]
comparacao["nome_bairro"] = comparacao.index.map(dict(zip(bairros["id_bairro_texto"], bairros["nome"])))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
comparacao.nlargest(15, "chamados").set_index("nome_bairro")["chamados"].sort_values().plot(kind="barh", ax=axes[0], color="forestgreen")
axes[0].set_title("Mais chamados (número total)")
comparacao.nlargest(15, "chamados_por_100_trechos").set_index("nome_bairro")["chamados_por_100_trechos"].sort_values().plot(kind="barh", ax=axes[1], color="darkorange")
axes[1].set_title("Mais chamados a cada 100 trechos de rua")
plt.tight_layout()
plt.show()

## 5.3 O id_logradouro dos chamados existe na tabela de logradouros?

In [ ]:
ids_logradouro = set(logradouros["id_logradouro"].astype(str))
tem_logradouro = chamados["id_logradouro"].notnull()
encontrado = chamados.loc[tem_logradouro, "id_logradouro"].astype(str).isin(ids_logradouro)
print(f"Chamados com id_logradouro preenchido: {tem_logradouro.sum()}")
print(f"Desses, quantos existem na tabela de logradouros: {100 * encontrado.mean():.1f}%")

## 5.4 Primeira comparação: chuva x chamados por mês
Coloco lado a lado a chuva do mês e os chamados de árvore do mês.

**Cuidado:** isto é só uma primeira olhada. Juntar por mês esconde o que acontece dia a dia (por exemplo, uma árvore que cai no dia seguinte a um temporal). E mesmo que as duas coisas subam juntas, isso não prova que uma causa a outra.

In [ ]:
juntos = pd.DataFrame({"chamados": por_mes, "chuva_mm": chuva_mes}).dropna()
datas = juntos.index.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].bar(datas, juntos["chuva_mm"], width=20, color="steelblue")
axes[0].set_title("Chuva no mês (mm, média das estações)")
axes[1].plot(datas, juntos["chamados"], color="forestgreen", marker="o")
axes[1].set_title("Chamados de Manejo Arbóreo no mês")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.scatter(juntos["chuva_mm"], juntos["chamados"], color="purple")
plt.xlabel("Chuva no mês (mm)")
plt.ylabel("Chamados no mês")
plt.title("Cada ponto é um mês")
plt.show()

print("Correlação entre chuva e chamados no mesmo mês:", round(juntos["chamados"].corr(juntos["chuva_mm"]), 2))

A **correlação** vai de -1 a 1. Perto de 1, as duas coisas sobem juntas; perto de 0, não há relação clara; perto de -1, quando uma sobe a outra desce.

---
# 6. Resumo

## O que corrigi em relação aos primeiros notebooks
- **Chamados:** consertei a consulta de bairros sem nome (vírgula faltando e `id.logradouro`) e tirei o mês incompleto do gráfico, que parecia uma queda de chamados.
- **Chuva:** o notebook anterior não tinha rodado e tinha partes repetidas. Troquei a média do acumulado de 24 horas pela soma das medições de 15 minutos por dia.
- **Logradouros:** entendi que cada linha é um trecho e criei uma tabela com uma linha por rua para não duplicar chamados.

## Dúvidas para levar ao grupo ou ao orientador
1. O que significam "A vencer" e "Em vencimento" para chamados que já foram encerrados?
2. Os valores negativos de chuva são mesmo códigos de erro?
3. Devo incluir chamados de árvore que estão em outros tipos além de "Manejo Arbóreo"?
4. Qual estação de chuva usar para cada bairro?

## Próximos passos
1. Definir o filtro final dos chamados: Manejo Arbóreo, categorias Serviço e Reclamação, e os grupos de subtipo (principalmente "Árvore caída" e "Avaliação de risco", que imagino que sejam os mais afetados por temporais).
2. Montar uma tabela **por dia**, com chamados e chuva, para olhar os dias seguintes a um temporal.
3. Ligar cada chamado à estação de chuva mais próxima. Para isso preciso descobrir se a tabela de chamados tem latitude e longitude (verificar na lista de colunas da seção 1.1).
4. Medir o tamanho dos bairros pelo comprimento das ruas, e não só pelo número de trechos.